In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [ ]:
# Winning RNN architecture (DO NOT CHANGE)
class RNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, 64, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(64, 64)
        self.out = nn.Linear(64, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))

class WeightedL1(nn.Module):
    def forward(self, pred, target):
        weights = 1.0 + 1.0 * torch.clamp(torch.abs(target) - 2.0, min=0)
        return (torch.abs(pred - target) * weights).mean()

def predict(model, loader):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(x.to(device)).cpu().numpy() for x, _ in loader]).flatten()

SEEDS = [42, 2024]
EXPECTED_TEST_SIZE = None  # Will be set after first prediction
print("Components ready")

✅ Components ready


In [ ]:
# Load standard data
train_loader, val_loader, pipeline = load_data(
    train_path="../datasets_challenge/train",
    train_ratio=0.98, batchsize=64,
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20, threshold=True, threshold_value=3.0,
    threshold_column="vxCG", target="sideSlip", scaling=True, seed=42,
)
test_loader = load_test_data("../datasets_challenge/test", 512, pipeline, load_anonymized_data=True)
input_size = next(iter(train_loader))[0].shape[2]

# Extract raw tensors for custom sampling
X_train_list, y_train_list = [], []
for x, y in train_loader:
    X_train_list.append(x)
    y_train_list.append(y)
X_train = torch.cat(X_train_list, dim=0)
y_train = torch.cat(y_train_list, dim=0)

X_val_list, y_val_list = [], []
for x, y in val_loader:
    X_val_list.append(x)
    y_val_list.append(y)
X_val = torch.cat(X_val_list, dim=0)
y_val = torch.cat(y_val_list, dim=0)

print(f"Data loaded | Input: {input_size} | Train: {len(X_train)} | Val: {len(X_val)}")
print(f"   Few-shot (|y|>=2): Train={( torch.abs(y_train)>=2).sum().item()}, Val={(torch.abs(y_val)>=2).sum().item()}")

../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv

../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.csv
../datasets_challenge/train/train_part_014.csv
../datasets_challenge/train/train_part_015.csv
../datasets_c

In [ ]:
def save_predictions(preds, filename):
    global EXPECTED_TEST_SIZE
    if EXPECTED_TEST_SIZE is None:
        EXPECTED_TEST_SIZE = len(preds)
    assert len(preds) == EXPECTED_TEST_SIZE, f"ERROR: Expected {EXPECTED_TEST_SIZE}, got {len(preds)}"
    df = pd.DataFrame({"id": range(len(preds)), "predicted_sideslip": preds})
    df.to_csv(filename, index=False)
    print(f"Saved: {filename} | Shape: {len(preds)} | Range: [{preds.min():.3f}, {preds.max():.3f}]")

def train_baseline(seed):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    for _ in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state, patience = val, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10: break
    model.load_state_dict(state)
    return model

def train_two_stage(seed, custom_train_loader=None):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    tl = custom_train_loader if custom_train_loader else train_loader
    
    # Stage 1
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    for _ in range(30):
        model.train()
        for x, y in tl:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    
    # Stage 2
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    crit2 = WeightedL1()
    best2, state2 = np.inf, None
    for _ in range(40):
        model.train()
        for x, y in tl:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    return model

print("Training functions ready")

✅ Training functions ready


## Approach 1: Oversampling Few-Shot Samples
Physically repeat |sideslip| >= 2 samples in training data

In [ ]:
print("="*60)
print("APPROACH 1: OVERSAMPLING FEW-SHOT")
print("="*60)

# Create oversampled dataset
few_shot_mask = torch.abs(y_train) >= 2.0
X_few = X_train[few_shot_mask]
y_few = y_train[few_shot_mask]

OVERSAMPLE_FACTOR = 5
X_oversampled = torch.cat([X_train] + [X_few] * OVERSAMPLE_FACTOR, dim=0)
y_oversampled = torch.cat([y_train] + [y_few] * OVERSAMPLE_FACTOR, dim=0)

# Shuffle
perm = torch.randperm(len(X_oversampled))
X_oversampled = X_oversampled[perm]
y_oversampled = y_oversampled[perm]

oversample_loader = DataLoader(TensorDataset(X_oversampled, y_oversampled), batch_size=64, shuffle=True)
print(f"Original: {len(X_train)} | Oversampled: {len(X_oversampled)} ({OVERSAMPLE_FACTOR}x few-shot)")

# Train
base_preds = [predict(train_baseline(s), test_loader) for s in SEEDS]
ts_preds = [predict(train_two_stage(s, oversample_loader), test_loader) for s in SEEDS]
blend = (np.mean(base_preds, axis=0) + np.mean(ts_preds, axis=0)) / 2
save_predictions(blend, "pred_v12_oversample5x.csv")

 APPROACH 1: OVERSAMPLING FEW-SHOT
Original: 134339 | Oversampled: 215019 (5x few-shot)
✅ pred_v12_oversample5x.csv | Shape: 49328 | Range: [-9.518, 9.695]
✅ pred_v12_oversample5x.csv | Shape: 49328 | Range: [-9.518, 9.695]


## Approach 2: Noise Augmentation for Few-Shot
Add Gaussian noise to few-shot samples to create synthetic data

In [ ]:
print("\n" + "="*60)
print("APPROACH 2: NOISE AUGMENTATION")
print("="*60)

def augment_with_noise(X, y, noise_std=0.05, n_augments=3):
    few_shot_mask = torch.abs(y) >= 2.0
    X_few = X[few_shot_mask]
    y_few = y[few_shot_mask]
    
    augmented_X = [X]
    augmented_y = [y]
    
    for _ in range(n_augments):
        noise = torch.randn_like(X_few) * noise_std
        augmented_X.append(X_few + noise)
        augmented_y.append(y_few)  # Keep same target
    
    return torch.cat(augmented_X, dim=0), torch.cat(augmented_y, dim=0)

X_aug, y_aug = augment_with_noise(X_train, y_train, noise_std=0.05, n_augments=4)
perm = torch.randperm(len(X_aug))
X_aug, y_aug = X_aug[perm], y_aug[perm]

noise_loader = DataLoader(TensorDataset(X_aug, y_aug), batch_size=64, shuffle=True)
print(f"Original: {len(X_train)} | With noise augmentation: {len(X_aug)}")

# Train
ts_preds = [predict(train_two_stage(s, noise_loader), test_loader) for s in SEEDS]
blend = (np.mean(base_preds, axis=0) + np.mean(ts_preds, axis=0)) / 2
save_predictions(blend, "pred_v12_noise_aug.csv")


 APPROACH 2: NOISE AUGMENTATION
Original: 134339 | With noise augmentation: 198883
Original: 134339 | With noise augmentation: 198883
✅ pred_v12_noise_aug.csv | Shape: 49328 | Range: [-9.219, 9.523]
✅ pred_v12_noise_aug.csv | Shape: 49328 | Range: [-9.219, 9.523]


## Approach 3: Focal Loss for Regression
Downweight easy (many-shot) samples dynamically

In [ ]:
print("\n" + "="*60)
print("APPROACH 3: FOCAL LOSS FOR REGRESSION")
print("="*60)

class FocalL1Loss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    
    def forward(self, pred, target):
        mae = torch.abs(pred - target)
        # Larger error = higher weight (focus on hard samples)
        # Normalize by max possible error to keep weights reasonable
        weight = (mae / (mae.max() + 1e-6)) ** self.gamma
        return (mae * (1 + weight)).mean()

def train_with_focal(seed, gamma=2.0):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    
    # Stage 1 with Focal
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = FocalL1Loss(gamma=gamma)
    best, state = np.inf, None
    for _ in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    
    # Stage 2 with WeightedL1 (proven)
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    crit2 = WeightedL1()
    best2, state2 = np.inf, None
    for _ in range(40):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    return model

# Test gamma=1.5 and gamma=2.0
for gamma in [1.5, 2.0]:
    print(f"\n  Focal gamma={gamma}")
    ts_preds = [predict(train_with_focal(s, gamma), test_loader) for s in SEEDS]
    blend = (np.mean(base_preds, axis=0) + np.mean(ts_preds, axis=0)) / 2
    save_predictions(blend, f"pred_v12_focal_g{gamma}.csv")


 APPROACH 3: FOCAL LOSS FOR REGRESSION

  Focal gamma=1.5
✅ pred_v12_focal_g1.5.csv | Shape: 49328 | Range: [-9.358, 9.714]

  Focal gamma=2.0
✅ pred_v12_focal_g1.5.csv | Shape: 49328 | Range: [-9.358, 9.714]

  Focal gamma=2.0
✅ pred_v12_focal_g2.0.csv | Shape: 49328 | Range: [-9.378, 9.577]
✅ pred_v12_focal_g2.0.csv | Shape: 49328 | Range: [-9.378, 9.577]


## Approach 4: Quantile Regression
Focus on upper tail of distribution

In [ ]:
print("\n" + "="*60)
print("APPROACH 4: QUANTILE REGRESSION")
print("="*60)

class QuantileLoss(nn.Module):
    def __init__(self, quantile=0.5):
        super().__init__()
        self.q = quantile
    
    def forward(self, pred, target):
        errors = target - pred
        return torch.max(self.q * errors, (self.q - 1) * errors).mean()

def train_quantile(seed, quantile=0.5):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = QuantileLoss(quantile=quantile)
    best, state = np.inf, None
    
    for _ in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    return model

# Standard median (q=0.5) as control, then try q=0.6, 0.7
for q in [0.5, 0.6, 0.7]:
    print(f"\n  Quantile={q}")
    q_preds = [predict(train_quantile(s, q), test_loader) for s in SEEDS]
    q_ens = np.mean(q_preds, axis=0)
    # Blend with standard baseline
    blend = (np.mean(base_preds, axis=0) + q_ens) / 2
    save_predictions(blend, f"pred_v12_quantile_{int(q*100)}.csv")


 APPROACH 4: QUANTILE REGRESSION

  Quantile=0.5
✅ pred_v12_quantile_50.csv | Shape: 49328 | Range: [-9.261, 9.623]

  Quantile=0.6
✅ pred_v12_quantile_50.csv | Shape: 49328 | Range: [-9.261, 9.623]

  Quantile=0.6
✅ pred_v12_quantile_60.csv | Shape: 49328 | Range: [-9.412, 9.719]

  Quantile=0.7
✅ pred_v12_quantile_60.csv | Shape: 49328 | Range: [-9.412, 9.719]

  Quantile=0.7
✅ pred_v12_quantile_70.csv | Shape: 49328 | Range: [-9.231, 9.731]
✅ pred_v12_quantile_70.csv | Shape: 49328 | Range: [-9.231, 9.731]


## Approach 5: Log-Transform Target
Compress extreme values during training

In [ ]:
print("\n" + "="*60)
print("APPROACH 5: LOG-TRANSFORM TARGET")
print("="*60)

def log_transform(y):
    return torch.sign(y) * torch.log1p(torch.abs(y))

def inv_log_transform(y):
    return torch.sign(y) * (torch.exp(torch.abs(y)) - 1)

# Transform targets
y_train_log = log_transform(y_train)
y_val_log = log_transform(y_val)

log_train_loader = DataLoader(TensorDataset(X_train, y_train_log), batch_size=64, shuffle=True)
log_val_loader = DataLoader(TensorDataset(X_val, y_val_log), batch_size=64, shuffle=False)

print(f"Original y range: [{y_train.min():.2f}, {y_train.max():.2f}]")
print(f"Log y range: [{y_train_log.min():.2f}, {y_train_log.max():.2f}]")

def train_log_target(seed):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    
    # Stage 1
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    for _ in range(30):
        model.train()
        for x, y in log_train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in log_val_loader) / len(log_val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    
    # Stage 2
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    crit2 = WeightedL1()
    best2, state2 = np.inf, None
    for _ in range(40):
        model.train()
        for x, y in log_train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in log_val_loader) / len(log_val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    return model

def predict_log(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in loader:
            pred_log = model(x.to(device))
            pred = inv_log_transform(pred_log)  # Transform back
            preds.append(pred.cpu().numpy())
    return np.concatenate(preds).flatten()

log_ts_preds = [predict_log(train_log_target(s), test_loader) for s in SEEDS]
log_ens = np.mean(log_ts_preds, axis=0)
blend = (np.mean(base_preds, axis=0) + log_ens) / 2
save_predictions(blend, "pred_v12_log_target.csv")


 APPROACH 5: LOG-TRANSFORM TARGET
Original y range: [-14.21, 11.65]
Log y range: [-2.72, 2.54]
✅ pred_v12_log_target.csv | Shape: 49328 | Range: [-10.466, 12.105]
✅ pred_v12_log_target.csv | Shape: 49328 | Range: [-10.466, 12.105]


## Approach 6: Weighted Random Sampler
Sample batches with higher probability for few-shot

In [ ]:
print("\n" + "="*60)
print("APPROACH 6: WEIGHTED RANDOM SAMPLER")
print("="*60)

# Create sample weights
sample_weights = torch.ones(len(y_train))
few_shot_mask = torch.abs(y_train) >= 2.0
sample_weights[few_shot_mask] = 5.0  # 5x more likely to be sampled

sampler = WeightedRandomSampler(sample_weights, num_samples=len(y_train), replacement=True)
weighted_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, sampler=sampler)

print(f"Few-shot samples: {few_shot_mask.sum().item()} ({100*few_shot_mask.float().mean():.1f}%)")
print(f"With 5x weight, expected ~{100*5*few_shot_mask.float().mean()/(1+4*few_shot_mask.float().mean()):.1f}% few-shot per epoch")

ts_preds = [predict(train_two_stage(s, weighted_loader), test_loader) for s in SEEDS]
blend = (np.mean(base_preds, axis=0) + np.mean(ts_preds, axis=0)) / 2
save_predictions(blend, "pred_v12_weighted_sampler.csv")


 APPROACH 6: WEIGHTED RANDOM SAMPLER
Few-shot samples: 16136 (12.0%)
With 5x weight, expected ~40.6% few-shot per epoch
✅ pred_v12_weighted_sampler.csv | Shape: 49328 | Range: [-9.459, 9.607]
✅ pred_v12_weighted_sampler.csv | Shape: 49328 | Range: [-9.459, 9.607]


## Approach 7: Two Expert Models
Separate models for small vs large sideslip, blend predictions

In [ ]:
print("\n" + "="*60)
print("APPROACH 7: TWO EXPERT MODELS")
print("="*60)

# Split data
small_mask = torch.abs(y_train) < 1.5
large_mask = torch.abs(y_train) >= 1.0  # Overlap intentional

X_small, y_small = X_train[small_mask], y_train[small_mask]
X_large, y_large = X_train[large_mask], y_train[large_mask]

small_loader = DataLoader(TensorDataset(X_small, y_small), batch_size=64, shuffle=True)
large_loader = DataLoader(TensorDataset(X_large, y_large), batch_size=64, shuffle=True)

print(f"Small expert data: {len(X_small)} | Large expert data: {len(X_large)}")

def train_expert(seed, loader):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    for _ in range(50):
        model.train()
        for x, y in loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    return model

# Train experts
small_preds = [predict(train_expert(s, small_loader), test_loader) for s in SEEDS]
large_preds = [predict(train_expert(s, large_loader), test_loader) for s in SEEDS]

small_ens = np.mean(small_preds, axis=0)
large_ens = np.mean(large_preds, axis=0)

# Simple blend: use large expert more when predictions are extreme
avg_pred = (small_ens + large_ens) / 2
large_weight = np.clip(np.abs(avg_pred) / 2.0, 0.3, 0.7)  # 30-70% large expert based on magnitude
expert_blend = small_ens * (1 - large_weight) + large_ens * large_weight

# Final blend with baseline
blend = (np.mean(base_preds, axis=0) + expert_blend) / 2
save_predictions(blend, "pred_v12_two_experts.csv")


 APPROACH 7: TWO EXPERT MODELS
Small expert data: 109059 | Large expert data: 40441
✅ pred_v12_two_experts.csv | Shape: 49328 | Range: [-8.827, 8.637]
✅ pred_v12_two_experts.csv | Shape: 49328 | Range: [-8.827, 8.637]


## Approach 8: Huber Loss (Smooth L1)
Less sensitive to outliers than L1

In [ ]:
print("\n" + "="*60)
print("APPROACH 8: HUBER LOSS")
print("="*60)

def train_huber(seed, delta=1.0):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    
    # Stage 1 with Huber
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.HuberLoss(delta=delta)
    best, state = np.inf, None
    for _ in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    
    # Stage 2 with WeightedL1
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    crit2 = WeightedL1()
    best2, state2 = np.inf, None
    for _ in range(40):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    return model

for delta in [0.5, 1.0, 2.0]:
    print(f"\n  Huber delta={delta}")
    ts_preds = [predict(train_huber(s, delta), test_loader) for s in SEEDS]
    blend = (np.mean(base_preds, axis=0) + np.mean(ts_preds, axis=0)) / 2
    save_predictions(blend, f"pred_v12_huber_d{delta}.csv")


 APPROACH 8: HUBER LOSS

  Huber delta=0.5
✅ pred_v12_huber_d0.5.csv | Shape: 49328 | Range: [-9.203, 9.702]

  Huber delta=1.0
✅ pred_v12_huber_d0.5.csv | Shape: 49328 | Range: [-9.203, 9.702]

  Huber delta=1.0
✅ pred_v12_huber_d1.0.csv | Shape: 49328 | Range: [-9.170, 9.761]

  Huber delta=2.0
✅ pred_v12_huber_d1.0.csv | Shape: 49328 | Range: [-9.170, 9.761]

  Huber delta=2.0
✅ pred_v12_huber_d2.0.csv | Shape: 49328 | Range: [-9.081, 9.529]
✅ pred_v12_huber_d2.0.csv | Shape: 49328 | Range: [-9.081, 9.529]
